**1. Verify Bronze files exist**

In [1]:
files = mssparkutils.fs.ls("Files/bronze/")
for f in files:
    print(f.name, f.size)

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 3, Finished, Available, Finished, False)

crypto_20260903_105402.json 78657
crypto_20260903_105616.json 78658
crypto_20260909_055643.json 79290
crypto_20260909_063321.json 78963
crypto_20260909_065323.json 78794


**2. Read all Bronze snapshots**

In [2]:
df_bronze = spark.read.option("multiline", "true").json("Files/bronze/*.json")
df_bronze.select("last_updated").distinct().show()

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 4, Finished, Available, Finished, False)

+--------------------+
|        last_updated|
+--------------------+
|2026-09-09T05:53:...|
|2026-09-09T06:30:...|
|2026-09-09T06:50:...|
|2026-09-03T10:53:...|
|2026-09-03T06:54:...|
|2026-09-03T10:51:...|
+--------------------+



**3. Get current Silver watermark (skip if Silver table doesn't exist yet)**

In [3]:
existing_max = spark.sql("SELECT MAX(LastUpdated) AS max_ts FROM silver_crypto_snapshot").collect()[0]["max_ts"]
print(existing_max)

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 5, Finished, Available, Finished, False)

2026-09-09 06:30:20


**4. Transform Bronze -> Silver shape, filter to new rows only**

In [4]:
from pyspark.sql.functions import col, to_timestamp

df_silver_new = df_bronze.select(
    col("id"), col("symbol"), col("name"), col("current_price"),
    col("market_cap_rank"), col("total_volume"),
    col("price_change_percentage_24h"), col("ath"),
    to_timestamp(col("ath_date")).alias("AthDate"),
    to_timestamp(col("last_updated")).alias("LastUpdated"),
    col("roi.percentage").alias("RoiPercentage")
).filter(col("LastUpdated") > existing_max)

df_silver_new = df_silver_new.dropDuplicates(["id", "LastUpdated"])
print(df_silver_new.count())

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 6, Finished, Available, Finished, False)

100


**5. Append new rows to Silver (never overwrite from here on)**

In [5]:
df_silver_new.write.format("delta").mode("append").saveAsTable("silver_crypto_snapshot")

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 7, Finished, Available, Finished, False)

**6. Audit trail**

In [6]:
from pyspark.sql import Row
from datetime import datetime

audit_row = Row(
    RunTimestamp=datetime.utcnow(),
    WatermarkBefore=str(existing_max),
    WatermarkAfter=str(df_silver_new.agg({"LastUpdated": "max"}).collect()[0][0]),
    RowsAppended=df_silver_new.count()
)

df_audit = spark.createDataFrame([audit_row])
df_audit.write.format("delta").mode("append").saveAsTable("audit_silver_runs")

display(spark.sql("SELECT * FROM audit_silver_runs ORDER BY RunTimestamp DESC"))

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5283b145-aa0e-4d4a-80b6-e6c9ee96782b)

**7. Verify Silver state**

In [7]:
spark.sql("SELECT MAX(LastUpdated) AS current_silver_max, COUNT(*) AS row_count FROM silver_crypto_snapshot").show()

StatementMeta(, 8a8f8110-11c2-429f-86a7-43e676e484ad, 9, Finished, Available, Finished, False)

+-------------------+---------+
| current_silver_max|row_count|
+-------------------+---------+
|2026-09-09 06:50:30|      499|
+-------------------+---------+



**8. Merge Logic A**

In [2]:
from pyspark.sql.functions import current_timestamp, lit, col
from delta.tables import DeltaTable

latest_state = spark.sql("""
    SELECT id, symbol, name, market_cap_rank
    FROM silver_crypto_snapshot
    WHERE LastUpdated = (SELECT MAX(LastUpdated) FROM silver_crypto_snapshot)
""")

if not spark.catalog.tableExists("dim_coin"):
    dim_coin_initial = latest_state \
        .withColumn("EffectiveStartDate", current_timestamp()) \
        .withColumn("EffectiveEndDate", lit(None).cast("timestamp")) \
        .withColumn("IsCurrent", lit(True))
    dim_coin_initial.write.format("delta").saveAsTable("dim_coin")
    print("dim_coin created with", dim_coin_initial.count(), "rows")
else:
    print("dim_coin already exists — run Cell B next")

StatementMeta(, dfd84f56-019b-40bb-928f-e6c1f9d91be1, 4, Finished, Available, Finished, False)

dim_coin already exists — run Cell B next


**8. Merge Logic B**

In [6]:
dim_coin_table = DeltaTable.forName(spark, "dim_coin")

# Find coins whose rank differs from their current active row
changed = latest_state.alias("new").join(
    spark.sql("SELECT * FROM dim_coin WHERE IsCurrent = true").alias("old"),
    col("new.id") == col("old.id")
).filter(
    col("new.market_cap_rank") != col("old.market_cap_rank")
).select("new.id")

changed_ids = [r["id"] for r in changed.collect()]
print("Coins with changed rank:", changed_ids)

# Close out their old rows
if changed_ids:
    dim_coin_table.update(
        condition = (col("IsCurrent") == True) & (col("id").isin(changed_ids)),
        set = {
            "IsCurrent": lit(False),
            "EffectiveEndDate": current_timestamp()
        }
    )

# Find genuinely new coins never seen before
existing_ids = [r["id"] for r in spark.sql("SELECT DISTINCT id FROM dim_coin").collect()]

new_or_changed = latest_state.filter(
    col("id").isin(changed_ids) | (~col("id").isin(existing_ids))
)

if new_or_changed.count() > 0:
    new_or_changed_final = new_or_changed \
        .withColumn("EffectiveStartDate", current_timestamp()) \
        .withColumn("EffectiveEndDate", lit(None).cast("timestamp")) \
        .withColumn("IsCurrent", lit(True))
    new_or_changed_final.write.format("delta").mode("append").saveAsTable("dim_coin")
    print("Inserted", new_or_changed_final.count(), "new/updated rows")
else:
    print("No changes detected — nothing inserted")

StatementMeta(, dfd84f56-019b-40bb-928f-e6c1f9d91be1, 8, Finished, Available, Finished, False)

Coins with changed rank: []
No changes detected — nothing inserted


**Test**

In [5]:
# TEST ONLY: artificially set bitcoin's stored rank to something wrong
dim_coin_table.update(
    condition = (col("id") == "bitcoin") & (col("IsCurrent") == True),
    set = { "market_cap_rank": lit(999) }
)

StatementMeta(, cb7bc131-8a47-4260-8818-4af8dd4acf09, 7, Finished, Available, Finished, False)

In [5]:
spark.sql("SELECT id, market_cap_rank, EffectiveStartDate, EffectiveEndDate, IsCurrent FROM dim_coin WHERE id = 'bitcoin' ORDER BY EffectiveStartDate").show(truncate=False)

StatementMeta(, dfd84f56-019b-40bb-928f-e6c1f9d91be1, 7, Finished, Available, Finished, False)

+-------+---------------+--------------------------+----------------+---------+
|id     |market_cap_rank|EffectiveStartDate        |EffectiveEndDate|IsCurrent|
+-------+---------------+--------------------------+----------------+---------+
|bitcoin|1              |2026-09-09 09:50:52.435979|NULL            |true     |
+-------+---------------+--------------------------+----------------+---------+



In [4]:
dim_coin_table.delete(condition = (col("id") == "bitcoin") & (col("market_cap_rank") == 999))

StatementMeta(, dfd84f56-019b-40bb-928f-e6c1f9d91be1, 6, Finished, Available, Finished, False)

**Gold-fact_price_snapshot**

In [8]:
fact_price_snapshot = spark.sql("""
    SELECT
        id AS CoinID,
        LastUpdated,
        current_price AS CurrentPrice,
        total_volume AS TotalVolume,
        price_change_percentage_24h AS PriceChangePct24h,
        ath AS AllTimeHigh,
        AthDate
    FROM silver_crypto_snapshot
""")

fact_price_snapshot.write.format("delta").mode("overwrite").saveAsTable("fact_price_snapshot")

print(fact_price_snapshot.count())

StatementMeta(, dfd84f56-019b-40bb-928f-e6c1f9d91be1, 10, Finished, Available, Finished, False)

499


<mark>X</mark>Step 1 — add SurrogateKey to dim_coin, with the correct schema option this time:

In [4]:
from pyspark.sql.functions import monotonically_increasing_id

dim_coin_with_key = spark.sql("SELECT * FROM dim_coin") \
    .withColumn("SurrogateKey", monotonically_increasing_id())

dim_coin_with_key.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_coin")

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 6, Finished, Available, Finished, False)

Updated

In [10]:
from pyspark.sql.functions import monotonically_increasing_id, min as spark_min, lit

earliest_per_coin = spark.sql("""
    SELECT id, MIN(LastUpdated) AS FirstSeen
    FROM silver_crypto_snapshot
    GROUP BY id
""")

latest_state = spark.sql("""
    SELECT id, symbol, name, market_cap_rank
    FROM silver_crypto_snapshot
    WHERE LastUpdated = (SELECT MAX(LastUpdated) FROM silver_crypto_snapshot)
""")

dim_coin_clean = latest_state.join(earliest_per_coin, "id") \
    .withColumnRenamed("FirstSeen", "EffectiveStartDate") \
    .withColumn("EffectiveEndDate", lit(None).cast("timestamp")) \
    .withColumn("IsCurrent", lit(True)) \
    .withColumn("SurrogateKey", monotonically_increasing_id())

dim_coin_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_coin")

print(dim_coin_clean.count())

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 12, Finished, Available, Finished, False)

100


Step 2 — verify it actually landed before moving on:

In [5]:
spark.sql("DESCRIBE dim_coin").show(truncate=False)

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 7, Finished, Available, Finished, False)

+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|id                |string   |NULL   |
|symbol            |string   |NULL   |
|name              |string   |NULL   |
|market_cap_rank   |bigint   |NULL   |
|EffectiveStartDate|timestamp|NULL   |
|EffectiveEndDate  |timestamp|NULL   |
|IsCurrent         |boolean  |NULL   |
|SurrogateKey      |bigint   |NULL   |
+------------------+---------+-------+



Step 3 — the join, using the correct schema-overwrite option, and reading dim_coin fresh from the table (not a stale in-memory variable):

In [11]:
fact_with_key = spark.sql("""
    SELECT
        f.CoinID,
        f.LastUpdated,
        f.CurrentPrice,
        f.TotalVolume,
        f.PriceChangePct24h,
        f.AllTimeHigh,
        f.AthDate,
        d.SurrogateKey
    FROM fact_price_snapshot f
    JOIN dim_coin d
      ON f.CoinID = d.id
     AND f.LastUpdated >= d.EffectiveStartDate
     AND (f.LastUpdated < d.EffectiveEndDate OR d.EffectiveEndDate IS NULL)
""")

print(fact_with_key.count())

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 13, Finished, Available, Finished, False)

489


Test1

In [7]:
spark.sql("""
    SELECT COUNT(*) AS matching_ids
    FROM fact_price_snapshot f
    JOIN dim_coin d ON f.CoinID = d.id
""").show()

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 9, Finished, Available, Finished, False)

+------------+
|matching_ids|
+------------+
|         489|
+------------+



Test2

In [8]:
spark.sql("""
    SELECT f.CoinID, f.LastUpdated, d.EffectiveStartDate, d.EffectiveEndDate
    FROM fact_price_snapshot f
    JOIN dim_coin d ON f.CoinID = d.id
    WHERE f.CoinID = 'bitcoin'
""").show(truncate=False)

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 10, Finished, Available, Finished, False)

+-------+-------------------+--------------------------+----------------+
|CoinID |LastUpdated        |EffectiveStartDate        |EffectiveEndDate|
+-------+-------------------+--------------------------+----------------+
|bitcoin|2026-09-03 10:51:20|2026-09-09 09:50:52.435979|NULL            |
|bitcoin|2026-09-03 10:53:30|2026-09-09 09:50:52.435979|NULL            |
|bitcoin|2026-09-09 05:53:30|2026-09-09 09:50:52.435979|NULL            |
|bitcoin|2026-09-09 06:30:20|2026-09-09 09:50:52.435979|NULL            |
|bitcoin|2026-09-09 06:50:30|2026-09-09 09:50:52.435979|NULL            |
+-------+-------------------+--------------------------+----------------+



Overwrite

In [13]:
fact_with_key.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_price_snapshot")

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 15, Finished, Available, Finished, False)

Final Integrity Check

In [14]:
spark.sql("SELECT COUNT(*) AS total, COUNT(DISTINCT SurrogateKey) AS distinct_keys FROM fact_price_snapshot").show()

StatementMeta(, 3d82710f-2fd2-4468-8ac1-8af66d3d9d81, 16, Finished, Available, Finished, False)

+-----+-------------+
|total|distinct_keys|
+-----+-------------+
|  489|          100|
+-----+-------------+



Figure Heloc Check

In [1]:
spark.sql("SELECT * FROM fact_price_snapshot WHERE CoinID = 'figure-heloc'").show(truncate=False)

StatementMeta(, b07722fb-b385-43e8-a7fd-bdf876dafd7d, 3, Finished, Available, Finished, False)

+------------+-------------------+------------+-----------+-----------------+-----------+-------------------+------------+
|CoinID      |LastUpdated        |CurrentPrice|TotalVolume|PriceChangePct24h|AllTimeHigh|AthDate            |SurrogateKey|
+------------+-------------------+------------+-----------+-----------------+-----------+-------------------+------------+
|figure-heloc|2026-09-03 06:54:00|1.013       |6.514494E7 |-2.0048          |1.061      |2026-08-03 14:19:20|17179869215 |
|figure-heloc|2026-09-09 05:53:30|1.035       |3.4096114E7|NULL             |1.061      |2026-08-03 14:19:20|17179869215 |
|figure-heloc|2026-09-09 06:50:30|1.035       |3.4096747E7|NULL             |1.061      |2026-08-03 14:19:20|17179869215 |
|figure-heloc|2026-09-09 06:30:20|1.035       |3.4095943E7|NULL             |1.061      |2026-08-03 14:19:20|17179869215 |
+------------+-------------------+------------+-----------+-----------------+-----------+-------------------+------------+

